In [80]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix


In [81]:
df = pd.read_csv('final.csv')

In [82]:
df['Belde']=df['Belde'].astype('category')
df['Konum']=df['Konum'].astype('category')
df['Metrekare']=df['Metrekare'].astype('int')
df['Oda']=df['Oda'].astype('int')
df['Fiyat']=df['Fiyat'].astype('int')

In [83]:
kategoriler = ['Belde', 'Konum']
sayisal = ['Metrekare','Oda']  

In [84]:
full_pipeline = ColumnTransformer([
    ('num', StandardScaler(), sayisal),
    ('cat', OneHotEncoder(handle_unknown='ignore'), kategoriler)
])

In [74]:
#ESKİ KOD, bozuk değer var
bins = [x for x in range(0, 70000, 10000)]
labels = [x for x in range(1,7)]
print(bins)
print(labels)

[0, 10000, 20000, 30000, 40000, 50000, 60000]
[1, 2, 3, 4, 5, 6]


In [86]:
# X ve y'yi ayırmadan ÖNCE filtreleme yapıyoruz:
df = df[df['Fiyat'] <= 60000]

# Sonra X ve y'yi ayırıyoruz
X = df.drop('Fiyat', axis=1)
y = df['Fiyat']

# Artık bins aralığını güvenle kullanabilirsin
bins = [0, 10000, 20000, 30000, 40000, 50000, 60000]
labels = [1, 2, 3, 4, 5, 6]
y = pd.cut(y, bins=bins, labels=labels)

In [ ]:
# Aralıklar: 0-10k, 10k-20k, 20k ve üzeri GERÇEK()bu da olmadı()
bins = [0, 10000, 20000, np.inf]
labels = [1, 2, 3]

y = pd.cut(y, bins=bins, labels=labels)
print(y.unique()) # Artık NaN görmeyeceksin

[1, NaN]
Categories (3, int64): [1 < 2 < 3]


In [ ]:
y = pd.cut(y, bins=bins, labels=labels)
#sana gerek yok

In [87]:
print(y.unique())

[6, 4, 3, 5, 2, 1]
Categories (6, int64): [1 < 2 < 3 < 4 < 5 < 6]


In [88]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [89]:
model = Pipeline([
    ('preparation', full_pipeline),
    ('model', RandomForestClassifier(n_estimators=100))
])

In [90]:
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preparation', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers con

In [91]:
y_pred = model.predict(X_test)

In [92]:
print(confusion_matrix(y_test,  y_pred))

[[ 0  0  0  0  0  0]
 [ 0  6  4  0  0  0]
 [ 0  4 32 17  0  0]
 [ 2  1 15 17  5  4]
 [ 1  0  4  6 14  3]
 [ 0  1  2  1  2  8]]


In [93]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.00      0.00      0.00         0
           2       0.50      0.60      0.55        10
           3       0.56      0.60      0.58        53
           4       0.41      0.39      0.40        44
           5       0.67      0.50      0.57        28
           6       0.53      0.57      0.55        14

    accuracy                           0.52       149
   macro avg       0.45      0.44      0.44       149
weighted avg       0.53      0.52      0.52       149



/home/yusuf/pyhton_dersleri/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/yusuf/pyhton_dersleri/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/yusuf/pyhton_dersleri/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.sh